# 01b — Limpieza e ingeniería

Dedup, descriptiva, imputación, temporalidad, festivos e índices finales (bloque 2 del cuaderno 01). Lee `firms_spain_full_enrichment.csv`; escribe `expanded/wildfire_data_cleaned.csv`.

## Imports (celda única del cuaderno)

In [1]:
# Imports del cuaderno (celda única) — todas las dependencias aquí; las demás celdas no importan nada.
from IPython.display import display
import holidays
import numpy as np
import pandas as pd
from wildfire.config import load_config
from wildfire.data.io import load_csv, save_csv
from wildfire.processing.dedup_cross_sensor import apply_modeling_schema, collapse_cross_sensor_dupes


## 2.1 — Eliminación de duplicados entre sensores

Colapsa detecciones duplicadas VIIRS/MODIS y aplica el esquema de modelado; produce `df_no_dupe`.

In [2]:
# Eliminación de duplicados


df_no_dupe = apply_modeling_schema(
    collapse_cross_sensor_dupes(
        load_csv("data/processed/enriched/firms_spain_full_enrichment.csv")
    )
)


## 2.2 — Estadística descriptiva

Describe el `df_no_dupe` de la 2.1 (sin columnas de contexto); solo visualiza.

In [3]:
# Estadística descriptiva


num_years = load_config()["ccaa"]["budget_lookback_years"]

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)


DESCRIBE_DROP_COLUMNS = [
    "acq_date",
    "acq_time",
    "latitude",
    "longitude",
    "confidence_cat",
    "confidence_num",
    "daynight",
    "clc_class",
    "clc_class_N",
    "clc_class_S",
    "clc_class_W",
    "clc_class_E",
    "clc_class_NW",
    "clc_class_NE",
    "clc_class_SW",
    "clc_class_SE",
    "clc_uniform_surroundings",
    "ccaa",
    "type",
    "year",
    "is_day",
    "prevention_current",
    f"prevention_prev{num_years}",
    f"extinction_prev{num_years}"
]

df_prov_describe = df_no_dupe.drop(columns=DESCRIBE_DROP_COLUMNS)

print(df_prov_describe.columns)
display(df_prov_describe.describe())


Index(['brightness', 'scan', 'brightness_ir', 'frp', 'temperature_2m', 'precipitation', 'surface_pressure', 'relative_humidity_2m', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'rain', 'et0_fao_evapotranspiration', 'shortwave_radiation', 'boundary_layer_height', 'vapour_pressure_deficit', 'soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm'], dtype='str')


,brightness,scan,brightness_ir,frp,temperature_2m,precipitation,surface_pressure,relative_humidity_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,rain,et0_fao_evapotranspiration,shortwave_radiation,boundary_layer_height,vapour_pressure_deficit,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm
count,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,30414.000000,25486.000000,30414.000000,30414.000000,30414.000000
mean,328.438769,0.590530,291.874330,16.242903,17.907186,0.017980,947.934534,50.807691,11.650521,184.996930,31.275661,0.017967,0.264477,361.283877,822.789760,1.325466,0.217584,0.241485
std,20.220547,0.437088,13.651375,48.805678,8.277225,0.249588,48.138964,21.493757,7.356515,96.314224,16.201410,0.249580,0.258843,376.175085,734.439161,1.219891,0.112175,0.104077
min,207.930000,0.320000,217.700000,0.000000,-4.316667,0.000000,776.221182,3.331267,0.024000,1.219271,2.232000,0.000000,0.000000,0.000000,10.000000,0.000000,-0.000500,-0.001333
25%,310.050000,0.400000,283.190000,2.200000,11.600000,0.000000,919.243478,34.062847,6.160471,96.649636,18.751500,0.000000,0.028936,0.000000,176.000000,0.462294,0.124000,0.149000
50%,330.370000,0.460000,290.360000,4.740000,16.963333,0.000000,953.115325,50.015343,9.809385,197.249062,28.283998,0.000000,0.193873,360.983333,577.500000,0.904436,0.193000,0.214000
75%,342.100000,0.560000,298.757500,11.700000,23.384584,0.000000,986.524707,67.011651,16.112995,258.806771,41.374497,0.000000,0.480923,728.391667,1330.125000,1.807262,0.322917,0.336000
max,471.100000,4.800000,400.100000,1369.500000,42.565832,11.500000,1037.450854,100.000000,49.461017,360.000000,141.959997,11.500000,0.926705,1003.133333,3784.666667,7.606591,0.513600,0.504000


## 2.3 — Limpieza de filas sin CCAA ni presupuestos

Elimina filas sin región o sin sumas (Ceuta/Melilla); produce `df_nan_clean`.

In [4]:
# Limpieza de filas sin CCAA y sin presupuestos (Ceuta y Melilla)

display(df_no_dupe.isna().sum())

# Las filas sin ccaa o sin presupuestos (Ceuta/Melilla) se eliminan
# (ver recuentos de faltantes arriba).
df_nan_clean = df_no_dupe.dropna(subset=["ccaa", "prevention_current", f"prevention_prev{num_years}", f"extinction_prev{num_years}"])

latitude                         0
longitude                        0
brightness                       0
scan                             0
acq_date                         0
acq_time                         0
brightness_ir                    0
frp                              0
daynight                         0
type                             0
year                             0
confidence_num                   0
confidence_cat                   0
temperature_2m                   0
precipitation                    0
surface_pressure                 0
relative_humidity_2m             0
wind_speed_10m                   0
wind_direction_10m               0
wind_gusts_10m                   0
rain                             0
et0_fao_evapotranspiration       0
shortwave_radiation              0
boundary_layer_height         4928
is_day                           0
vapour_pressure_deficit          0
soil_moisture_0_to_7cm           0
soil_moisture_7_to_28cm          0
clc_class           

## 2.4 — Imputación de `boundary_layer_height`

Rellena los faltantes con la mediana; el recuento se verifica en la salida.

In [5]:
# Detección de valores nulos; asignación de valor a boundary_layer_height
# Los faltantes de boundary_layer_height (ver recuento arriba) se imputan
# con la mediana por ser una variable con atípicos.

df_nan_clean.fillna({"boundary_layer_height": df_nan_clean["boundary_layer_height"].median()}, inplace=True)

display(df_nan_clean.isna().sum())

print(df_nan_clean.shape)

latitude                      0
longitude                     0
brightness                    0
scan                          0
acq_date                      0
acq_time                      0
brightness_ir                 0
frp                           0
daynight                      0
type                          0
year                          0
confidence_num                0
confidence_cat                0
temperature_2m                0
precipitation                 0
surface_pressure              0
relative_humidity_2m          0
wind_speed_10m                0
wind_direction_10m            0
wind_gusts_10m                0
rain                          0
et0_fao_evapotranspiration    0
shortwave_radiation           0
boundary_layer_height         0
is_day                        0
vapour_pressure_deficit       0
soil_moisture_0_to_7cm        0
soil_moisture_7_to_28cm       0
clc_class                     0
clc_class_N                   0
clc_class_S                   0
clc_clas

(30412, 42)


## 2.5 — Detección de atípicos (IQR)

Regla de Tukey solo como informe; no elimina ni transforma nada.

In [6]:
# Detección de outliers
# Detección valores atipicos / Outlier detection (main dataframe is df_nan_clean)

df_prov_describe = df_nan_clean.drop(columns=DESCRIBE_DROP_COLUMNS)

display(df_prov_describe.describe())

# We apply the Turkey IQR rule
Q1, Q3 = df_prov_describe.quantile(0.25), df_prov_describe.quantile(0.75)
IQR = Q3 - Q1
outlier_counts = ((df_prov_describe < Q1 - 1.5 * IQR) | (df_prov_describe > Q3 + 1.5 * IQR)).sum()
print(outlier_counts[outlier_counts > 0])

# Nuestros outliers son: frp, scan, brightness, brightness_ir, surface_pressure, wind_speed_10m, wind_gusts_10m, boundary_layer_height, vapour_pressure_deficit
# Our outliers are: frp, scan, brightness, brightness_ir, surface_pressure, wind_speed_10m, wind_gusts_10m, boundary_layer_height, vapour_pressure_deficit

# temperature_2m solo tiene 52 valores que se salen de los percentiles, así que no son outliers significativos
# temperature_2m only has 52 values that fall outside the percentiles, so they are not significant outliers.

# precipitation y rain no son outliers ya que la mayoría de los días no llueve (0), por lo que parecen outliers aunque no lo sean
# precipitation and rain are not outliers since most days it does not rain (0), so they seem like outliers even though they are not

,brightness,scan,brightness_ir,frp,temperature_2m,precipitation,surface_pressure,relative_humidity_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,rain,et0_fao_evapotranspiration,shortwave_radiation,boundary_layer_height,vapour_pressure_deficit,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm
count,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000,30412.000000
mean,328.440597,0.590542,291.875231,16.243926,17.907660,0.017981,947.932580,50.804865,11.650425,185.004355,31.276006,0.017968,0.264494,361.307636,783.076324,1.325546,0.217581,0.241484
std,20.219915,0.437100,13.651346,48.807120,8.277115,0.249597,48.139575,21.491630,7.356720,96.312996,16.201837,0.249589,0.258843,376.176044,678.368234,1.219891,0.112178,0.104079
min,207.930000,0.320000,217.700000,0.000000,-4.316667,0.000000,776.221182,3.331267,0.024000,1.219271,2.232000,0.000000,0.000000,0.000000,10.000000,0.000000,-0.000500,-0.001333
25%,310.057500,0.400000,283.190000,2.200000,11.600000,0.000000,919.240936,34.061264,6.160471,96.663733,18.750000,0.000000,0.028946,0.000000,231.187500,0.462334,0.124000,0.149000
50%,330.370000,0.460000,290.360000,4.740000,16.963333,0.000000,953.112365,50.013003,9.809009,197.249062,28.283998,0.000000,0.193911,361.283333,577.500000,0.904505,0.193000,0.214000
75%,342.102500,0.560000,298.760000,11.700000,23.385625,0.000000,986.517642,67.006875,16.113452,258.806771,41.375998,0.000000,0.480938,728.425000,1180.833333,1.807262,0.322917,0.336000
max,471.100000,4.800000,400.100000,1369.500000,42.565832,11.500000,1037.450854,100.000000,49.461017,360.000000,141.959997,11.500000,0.926705,1003.133333,3784.666667,7.606591,0.513600,0.504000


brightness                   95
scan                       3618
brightness_ir               853
frp                        3789
temperature_2m               18
precipitation              1545
surface_pressure            447
wind_speed_10m              361
wind_gusts_10m              397
rain                       1544
boundary_layer_height       575
vapour_pressure_deficit    1760
dtype: int64


## 2.6 — Descomposición temporal

`acq_date` a datetime más `month`, `day_of_year` y `es_fin_semana`.

In [7]:
# DESCOMPOSICIÓN TEMPORAL
df_nan_clean['acq_date'] = pd.to_datetime(df_nan_clean['acq_date'])

df_nan_clean['month'] = df_nan_clean['acq_date'].dt.month
df_nan_clean['day_of_year'] = df_nan_clean['acq_date'].dt.dayofyear
df_nan_clean['es_fin_semana'] = df_nan_clean['acq_date'].dt.dayofweek.isin([5, 6]).astype(int)

## 2.7 — Tramos horarios de `acq_time`

Hora extraída de `acq_time` y recodificada en `franja_horaria`.

In [8]:
# Segmentación de acq_time en tramos del día
df_nan_clean['acq_time_str'] = df_nan_clean['acq_time'].astype(str).str.zfill(4)
df_nan_clean['hora'] = pd.to_datetime(df_nan_clean['acq_time_str'], format='%H%M', errors='coerce').dt.hour

bins_hora = [-1, 6, 12, 18, 24]
labels_hora = ['Madrugada', 'Mañana', 'Tarde_Pico_Calor', 'Noche']
df_nan_clean['franja_horaria'] = pd.cut(df_nan_clean['hora'], bins=bins_hora, labels=labels_hora, right=False)

## 2.8 — Festivos y riesgo antrópico

Cruza fecha×CCAA con el calendario (`holidays.Spain`) en `es_festivo` y `riesgo_antropico_alto`.

In [9]:
# Calcular festivos solo para combinaciones únicas de fecha/CCAA


# Festivos 
mapa_ccaa_holidays = {
    'Andalucía': 'AN', 'Aragón': 'AR', 'Asturias': 'AS', 'Islas Baleares': 'IB', 
    'Canarias': 'CN', 'Cantabria': 'CB', 'Castilla y León': 'CL', 
    'Castilla-La Mancha': 'CM', 'Cataluña': 'CT', 'Comunidad Valenciana': 'VC', 
    'Extremadura': 'EX', 'Galicia': 'GA', 'Madrid': 'MD', 
    'Región de Murcia': 'MC', 'Navarra': 'NC', 'País Vasco': 'PV', 'La Rioja': 'RI'
}

calendario_unico = df_nan_clean[['acq_date', 'ccaa']].dropna().drop_duplicates()

def evaluar_festivo(fila):
    codigo_region = mapa_ccaa_holidays.get(fila['ccaa'])
    if not codigo_region:
        return 0
    cal = holidays.Spain(years=fila['acq_date'].year, subdiv=codigo_region)
    return int(fila['acq_date'] in cal)

calendario_unico['es_festivo'] = calendario_unico.apply(evaluar_festivo, axis=1)

df_nan_clean = df_nan_clean.merge(calendario_unico, on=['acq_date', 'ccaa'], how='left')
df_nan_clean['es_festivo'] = df_nan_clean['es_festivo'].fillna(0).astype(int)

# Variable riesgo antrópico
df_nan_clean['riesgo_antropico_alto'] = ((df_nan_clean['es_fin_semana'] == 1) | (df_nan_clean['es_festivo'] == 1)).astype(int)


## 2.9 — Codificación cíclica

Seno/coseno de mes y día del año para la estacionalidad sin saltos.

In [10]:
# Codificación cíclica con sin/cos

df_nan_clean['month_sin'] = np.sin(2 * np.pi * df_nan_clean['month'] / 12)
df_nan_clean['month_cos'] = np.cos(2 * np.pi * df_nan_clean['month'] / 12)
df_nan_clean['day_of_year_sin'] = np.sin(2 * np.pi * df_nan_clean['day_of_year'] / 365.25)
df_nan_clean['day_of_year_cos'] = np.cos(2 * np.pi * df_nan_clean['day_of_year'] / 365.25)


## 2.10 — Hectáreas MITECO e índices finales

Ratios por hectárea e índices meteorológicos derivados; escribe `expanded/wildfire_data_cleaned.csv`.

In [ ]:
# Hectáreas forestales (MITECO)

num_years = load_config()["ccaa"]["budget_lookback_years"]

superficie_forestal_ha = {
    'Andalucía': 4400000, 'Aragón': 2600000, 'Asturias': 770000,
    'Islas Baleares': 220000, 'Canarias': 570000, 'Cantabria': 360000,
    'Castilla y León': 5100000, 'Castilla-La Mancha': 3600000, 'Cataluña': 2000000,
    'Comunidad Valenciana': 1300000, 'Extremadura': 2700000, 'Galicia': 2000000,
    'Madrid': 430000, 'Región de Murcia': 510000, 'Navarra': 610000,
    'País Vasco': 490000, 'La Rioja': 340000
}

df_nan_clean['sup_forestal_ha'] = df_nan_clean['ccaa'].map(superficie_forestal_ha)

df_nan_clean['ratio_prevencion_extincion'] = df_nan_clean[f'prevention_prev{num_years}'] / (df_nan_clean[f'extinction_prev{num_years}'] + 1e-5)
df_nan_clean['euros_prevencion_por_ha'] = (df_nan_clean[f'prevention_prev{num_years}'] * 1000000) / df_nan_clean['sup_forestal_ha']
df_nan_clean['euros_extincion_por_ha'] = (df_nan_clean[f'extinction_prev{num_years}'] * 1000000) / df_nan_clean['sup_forestal_ha']


# Potencial de secado atmosférico acelerado por el viento
df_nan_clean['potencial_propagacion'] = df_nan_clean['vapour_pressure_deficit'] * df_nan_clean['wind_speed_10m']

df_nan_clean['indice_riesgo_meteo'] = df_nan_clean['potencial_propagacion'] / (df_nan_clean['soil_moisture_0_to_7cm'] + 0.01)

df_nan_clean['estres_termico_vegetacion'] = df_nan_clean['temperature_2m'] / (df_nan_clean['relative_humidity_2m'] + 1)


df_nan_clean['indice_riesgo_meteo_log'] = np.log1p(df_nan_clean['indice_riesgo_meteo'])

# 7. Limpieza final
columnas_basura = [
    'acq_time_str', 'hora',
    'es_fin_semana', 'es_festivo'
]
df_nan_clean = df_nan_clean.drop(columns=columnas_basura)

save_csv(df_nan_clean, "data/processed/expanded/wildfire_data_cleaned.csv")

display(df_nan_clean)
print(df_nan_clean.shape)


,latitude,longitude,brightness,scan,acq_date,acq_time,brightness_ir,frp,daynight,type,year,confidence_num,confidence_cat,temperature_2m,precipitation,surface_pressure,relative_humidity_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,rain,et0_fao_evapotranspiration,shortwave_radiation,boundary_layer_height,is_day,vapour_pressure_deficit,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,clc_class,clc_class_N,clc_class_S,clc_class_W,clc_class_E,clc_class_NW,clc_class_NE,clc_class_SW,clc_class_SE,clc_uniform_surroundings,ccaa,prevention_current,prevention_prev3,extinction_prev3,month,day_of_year,franja_horaria,riesgo_antropico_alto,month_sin,month_cos,day_of_year_sin,day_of_year_cos,sup_forestal_ha,ratio_prevencion_extincion,euros_prevencion_por_ha,euros_extincion_por_ha,potencial_propagacion,indice_riesgo_meteo,estres_termico_vegetacion,indice_riesgo_meteo_log
0,43.12620,-4.09110,316.00,1.10,2023-01-01,221,273.70,20.40,N,0,2023,92,h,12.653500,0.0,958.578073,55.531887,19.605200,177.975278,66.833999,0.0,0.077642,0.000000,569.750000,0.0,0.652047,0.378650,0.384650,3,5,3,3,3,3,3,3,3,False,Cantabria,2.00,6.5,13.2,1,1,Madrugada,1,5.000000e-01,0.866025,0.017202,0.999852,360000,0.492424,18.055556,36.666667,12.783511,32.892090,0.223829,3.523182
1,39.20470,-1.76060,304.40,1.00,2023-01-01,1326,286.60,6.20,D,0,2023,59,n,15.485000,0.0,944.967391,50.933622,5.877980,88.486478,14.495999,0.0,0.240336,442.033333,442.333333,1.0,0.865419,0.207567,0.236000,5,5,5,5,5,5,5,5,5,True,Castilla-La Mancha,52.00,120.0,140.0,1,1,Tarde_Pico_Calor,1,5.000000e-01,0.866025,0.017202,0.999852,3600000,0.857143,33.333333,38.888889,5.086914,23.380942,0.298169,3.193802
2,37.99410,-4.32650,303.50,1.10,2023-01-01,1326,286.80,7.10,D,0,2023,53,n,17.718334,0.0,1003.219061,47.663541,3.478249,171.463845,12.995999,0.0,0.233387,408.133333,465.333333,1.0,1.063250,0.314000,0.317000,7,7,7,7,6,7,7,7,7,False,Andalucía,110.00,250.0,271.0,1,1,Tarde_Pico_Calor,1,5.000000e-01,0.866025,0.017202,0.999852,4400000,0.922509,56.818182,61.590909,3.698250,11.414351,0.364099,2.518853
3,42.38850,-2.72820,301.20,3.10,2023-01-03,1008,279.50,40.00,D,0,2023,45,n,6.254333,0.0,969.943058,82.101553,5.603572,300.106980,14.855999,0.0,0.079870,230.466667,235.333333,1.0,0.171449,0.341000,0.318000,5,5,5,5,5,5,5,5,5,True,La Rioja,2.54,6.3,12.3,1,3,Mañana,0,5.000000e-01,0.866025,0.051584,0.998669,340000,0.512195,18.529412,36.176471,0.960729,2.737120,0.075261,1.318315
4,38.59230,-1.07100,304.70,1.10,2023-01-03,1313,291.00,6.30,D,0,2023,53,n,15.236666,0.0,963.624625,59.581054,2.225724,135.059887,12.006000,0.0,0.227979,429.166667,849.083333,1.0,0.700881,0.036000,0.001000,5,5,5,5,5,5,5,5,5,True,Región de Murcia,5.80,14.5,31.0,1,3,Tarde_Pico_Calor,0,5.000000e-01,0.866025,0.051584,0.998669,510000,0.467742,28.431373,60.784314,1.559968,33.912346,0.251509,3.552841
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30407,38.96954,-5.91559,339.95,0.44,2024-12-31,1332,288.55,4.24,D,0,2024,50,n,12.233333,0.0,999.142452,56.079420,6.873135,56.769397,19.584000,0.0,0.217841,452.933333,435.333333,1.0,0.628065,0.374467,0.377000,7,7,7,7,7,7,7,7,7,True,Extremadura,33.00,80.0,123.0,12,366,Tarde_Pico_Calor,0,-2.449294e-16,1.000000,0.012901,0.999917,2700000,0.650406,29.629630,45.555556,4.316778,11.227965,0.214321,2.503726
30408,38.82438,-6.21968,327.26,0.43,2024-12-31,1332,288.10,2.13,D,0,2024,50,n,11.853333,0.0,997.890291,60.617241,11.270243,52.081457,25.199999,0.0,0.213924,457.466667,347.333333,1.0,0.548777,0.363467,0.383467,5,6,4,4,4,4,5,7,7,False,Extremadura,33.00,80.0,123.0,12,366,Tarde_Pico_Calor,0,-2.449294e-16,1.000000,0.012901,0.999917,2700000,0.650406,29.629630,45.555556,6.184850,16.560648,0.192370,2.865660
30409,39.35228,-6.20244,327.92,0.44,2024-12-31,1332,286.80,6.14,D,0,2024,50,n,11.213333,0.0,978.742855,63.191774,7.802388,31.149425,18.888

(30412, 58)


## 2.11 — Quemas agrícolas candidatas

El satélite no distingue quemas de rastrojo de incendios forestales (límite declarado en el plan no supervisado §2). Heurística reversible: **se apartan** las detecciones **dentro de la ventana** de quema controlada (15 oct–30 may, configurable por grupo de CCAA) **y** sobre combustibles herbáceos (`clc_class` 6 "Permanent herbaceous", 7 "Periodically herbaceous"): en meses fríos no hay régimen de incendio forestal sobre herbáceas, así que esas detecciones son casi con seguridad quemas gestionadas. Todo lo demás se conserva: en verano todo (cualquier clase), en invierno salvo herbáceas.

Parámetros editables arriba del código (grupos, ventanas, clases). La celda diagnostica antes de filtrar (conteos, FRP, mes) y falla si las sospechosas no tienen FRP menor que las conservadas. Escribe `expanded/wildfire_data_cleaned.csv` (conservadas) y `expanded/quemas_agricolas_candidatas.csv` (apartadas, para perfilar aparte).


In [12]:
# Quemas agrícolas candidatas: dentro de ventana (15oct-30may) y sobre herbáceas (CLC 6/7)

GRUPOS_CCAA = {
    "norte": ["Galicia", "Asturias", "Cantabria", "País Vasco", "Navarra", "La Rioja"],
    "centro": ["Castilla y León", "Aragón", "Cataluña", "Madrid", "Castilla-La Mancha", "Extremadura"],
    "sur": ["Andalucía", "Comunidad Valenciana", "Región de Murcia", "Islas Baleares", "Canarias"],
}
VENTANAS = {"norte": ((10, 15), (5, 30)), "centro": ((10, 15), (5, 30)), "sur": ((10, 15), (5, 30))}
CLC_QUEMA = [6, 7]

grupo = df_nan_clean["ccaa"].map({c: g for g, cs in GRUPOS_CCAA.items() for c in cs})
sin_grupo = sorted(df_nan_clean.loc[grupo.isna(), "ccaa"].unique())
print(f"CCAA sin grupo (se conservan siempre): {sin_grupo}")
mes, dia = df_nan_clean["month"], df_nan_clean["acq_date"].dt.day
ini_mes = grupo.map({g: v[0][0] for g, v in VENTANAS.items()})
ini_dia = grupo.map({g: v[0][1] for g, v in VENTANAS.items()})
fin_mes = grupo.map({g: v[1][0] for g, v in VENTANAS.items()})
fin_dia = grupo.map({g: v[1][1] for g, v in VENTANAS.items()})
dentro = (
    (mes > ini_mes) | ((mes == ini_mes) & (dia >= ini_dia))
    | (mes < fin_mes) | ((mes == fin_mes) & (dia <= fin_dia))
    | grupo.isna()
)
sospecha = dentro & df_nan_clean["clc_class"].isin(CLC_QUEMA)

print(f"Total: {len(df_nan_clean)} | dentro: {int(dentro.sum())} | fuera: {int((~dentro).sum())} | sospechosas: {int(sospecha.sum())}")
print(pd.crosstab(grupo.fillna("sin-grupo"), sospecha))
frp_ok, frp_sosp = df_nan_clean.loc[~sospecha, "frp"].median(), df_nan_clean.loc[sospecha, "frp"].median()
print(f"FRP mediana (conservadas vs sospechosas): {frp_ok} vs {frp_sosp}")
print("Mes de las sospechosas:")
print(df_nan_clean.loc[sospecha, "month"].value_counts().sort_index().to_string())

if int(sospecha.sum()) and not (frp_sosp < frp_ok):
    raise ValueError("La heurística no separa por FRP: revisar VENTANAS/CLC_QUEMA antes de filtrar")

quemas = df_nan_clean[sospecha].copy()
df_nan_clean = df_nan_clean[~sospecha].copy()
save_csv(quemas, "data/processed/expanded/quemas_agricolas_candidatas.csv")
save_csv(df_nan_clean, "data/processed/expanded/wildfire_data_cleaned.csv")
print(f"Conservadas: {df_nan_clean.shape} -> wildfire_data_cleaned.csv | Apartadas: {quemas.shape} -> quemas_agricolas_candidatas.csv")


CCAA sin grupo (se conservan siempre): []
Total: 30412 | dentro: 19026 | fuera: 11386 | sospechosas: 7270
col_0   False  True 
ccaa                
centro   8132   2917
norte    6808   2299
sur      8202   2054
FRP mediana (conservadas vs sospechosas): 5.21 vs 3.82
Mes de las sospechosas:
month
1      564
2     1335
3     2081
4     1067
5      517
10     468
11     653
12     585


Conservadas: (23142, 58) -> wildfire_data_cleaned.csv | Apartadas: (7270, 58) -> quemas_agricolas_candidatas.csv
